# Chapter 31
## ING Rhythms
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter31.ipynb)

## About this chapter

Interneuron gamma (ING) is an inhibitory network rhythm: a population of
mutually inhibitory cells fires in near-synchronous volleys because every
cell's next spike is gated by the same decaying inhibitory conductance
recovering from the last volley, rather than by any recurrent excitation.
Recurrent inhibition has the form

$$
I_{{\rm II},i} = g_{\rm II}\sum_j s_j\,(E_I-v_i),
$$

optionally supplemented by gap-junction currents $\sum_j c_{ij}(v_j-v_i)$
between I cells.

The examples move from a single self-inhibited cell (a one-cell timing
reference and its sensitivity to parameter perturbations), through abstract
inhibitory pulse-coupling maps that reduce the population's timing to a
fixed point and its stability, to full spiking-network simulations
(`ING_1`-`ING_10`) that vary heterogeneity, connection sparsity, and gap
junctions. The last two examples show ING entraining a population of
excitatory cells, i.e. whether the inhibitory rhythm creates a repeatable
window in which E cells can fire.

See [`chapter31.md`](chapter31.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from numba import njit
from numba.typed import List

## Synaptic Release Timing (shared by every example below)

Every synapse here uses the two-gate scheme from earlier chapters: a fast
rise gate $q$ that opens right after a spike and closes with time constant
`tau_d_q`, driving a slower gate $s$ (rise `tau_r`, decay `tau_d`) that is
the actual synaptic conductance. `tau_d_q_function` picks `tau_d_q` (by
bisection) so that $s$'s peak occurs at a prescribed `tau_peak` after the
presynaptic spike.

The `ING_1`-`ING_10` and entrainment network sims below integrate 100-500
coupled cells over tens of thousands of time steps, so their per-timestep
update loops are `@njit`-compiled (numba); everything else here (single
cells, abstract maps, splay-state initializers) is plain NumPy, matching
this repo's convention of only accelerating the genuinely slow inner
loops.

In [ ]:
@njit
def tau_peak_function(tau_d, tau_r, tau_d_q):
    """Time (from a delta-function pulse of transmitter release) at which
    the synaptic gate s peaks."""
    dt = 0.01
    dt05 = 0.5 * dt

    s = 0.0
    t = 0.0
    s_inc = math.exp(-t / tau_d_q) * (1.0 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old = t
        s_inc_old = s_inc
        s_tmp = s + dt05 * s_inc
        s_inc_tmp = math.exp(-(t + dt05) / tau_d_q) * (1.0 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt * s_inc_tmp
        t = t + dt
        s_inc = math.exp(-t / tau_d_q) * (1.0 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


@njit
def tau_d_q_function(tau_d, tau_r, tau_hat):
    """Release time constant tau_d_q so that tau_peak_function reproduces
    the prescribed tau_hat (bisection search)."""
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left *= 0.5
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2.0
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = 0.5 * (tau_d_q_left + tau_d_q_right)
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return 0.5 * (tau_d_q_left + tau_d_q_right)

## 1-Cell ING

A single WB-style inhibitory cell whose own spikes feed back through a
self-inhibitory synapse (`g_ii`). This is the one-cell timing reference for
ING: the period is set entirely by how fast the cell recovers from its own
inhibition.

In [ ]:
def alpha_h(v):
    return 0.35 * np.exp(-(v + 58.0) / 20.0)


def alpha_m(v):
    return 0.1 * (v + 35.0) / (1.0 - np.exp(-0.1 * (v + 35.0)))


def alpha_n(v):
    return -0.05 * (v + 34.0) / (np.exp(-0.1 * (v + 34.0)) - 1.0)


def beta_h(v):
    return 5.0 / (np.exp(-0.1 * (v + 28.0)) + 1.0)


def beta_m(v):
    return 4.0 * np.exp(-(v + 60.0) / 18.0)


def beta_n(v):
    return 0.625 * np.exp(-(v + 44.0) / 80.0)


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))


def simulate_1_cell_ing(i_ext=1.5, g_ii=0.5, v_rev_i=-75.0, tau_r_i=0.5, tau_peak_i=0.5,
                         tau_d_i=9.0, t_final=200.0, dt=0.01, c=1.0, g_k=9.0, g_Na=35.0,
                         g_l=0.1, v_k=-90.0, v_na=55.0, v_l=-65.0):
    """Single self-inhibited WB cell. Returns (t, v)."""
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    def derivative(x0, t):
        v, h, n, q, s = x0
        I_Na = g_Na * m_inf(v) ** 3 * h * (v - v_na)
        I_L = g_l * (v - v_l)
        I_K = g_k * n ** 4 * (v - v_k)
        I_syn = g_ii * s * (v_rev_i - v)

        dv = -I_Na - I_K - I_L + i_ext + I_syn
        dh = alpha_h(v) * (1 - h) - beta_h(v) * h
        dn = alpha_n(v) * (1 - n) - beta_n(v) * n
        dq = 0.5 * (1.0 + np.tanh(0.1 * v)) * 10.0 * (1 - q) - q / tau_dq_i
        ds = q * (1.0 - s) / tau_r_i - s / tau_d_i
        return [dv, dh, dn, dq, ds]

    x0 = [-75.0, 0.1, 0.1, 0.0, 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    v = sol[:, 0]
    return t, v


def plot_1_cell_ing(t, v):
    plt.figure(figsize=(7, 3))
    plt.plot(t, v, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.ylim(-100, 50)
    plt.xlabel("time [ms]")
    plt.ylabel("v [mV]")
    plt.yticks(range(-100, 100, 50))
    plt.tight_layout()
    plt.show()

In [ ]:
interact(lambda i_ext=1.5: plot_1_cell_ing(*simulate_1_cell_ing(i_ext=i_ext)),
         i_ext=(0.5, 3.0, 0.1));

## 1-Cell ING: Condition Numbers

Timing sensitivity of the one-cell reference: perturb `i_ext`, `g_ii`, or
`tau_d_i` by 1% and measure the resulting percentage change in the
oscillation period. Uses a WB tau/inf gating formulation (`m_i_inf`,
`h_i_inf`, `tau_h_i`, `n_i_inf`, `tau_n_i`) reused by every population
example below.

In [ ]:
def m_i_inf(v):
    alpha_m = 0.1 * (v + 35) / (1 - np.exp(-(v + 35) / 10))
    beta_m = 4. * np.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


def h_i_inf(v):
    alpha_h = 0.07 * np.exp(-(v + 58) / 20)
    beta_h = 1. / (np.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


def tau_h_i(v):
    alpha_h = 0.07 * np.exp(-(v + 58) / 20)
    beta_h = 1. / (np.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5


def n_i_inf(v):
    alpha_n = -0.01 * (v + 34) / (np.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * np.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


def tau_n_i(v):
    alpha_n = -0.01 * (v + 34) / (np.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * np.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5


def simulate_single_wb_ing(i_ext_i, g_ii, tau_d_i, v_rev_i=-75.0, tau_r_i=0.5, tau_peak_i=0.5,
                            t_final=200.0, dt=0.01, record_trace=False):
    """Single self-inhibited WB cell (tau/inf gating, explicit Heun
    stepping). Returns (period, v_trace or None)."""
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    v_i = [-75.]
    h_i, n_i, q_i, s_i = 0.1, 0.1, 0., 0.
    t_i_spikes = []

    for k in range(m_steps):
        t_old, t_new = k * dt, (k + 1) * dt
        vk = v_i[k]

        v_inc = (0.1 * (-65 - vk) + 9 * n_i ** 4 * (-90 - vk) + 35 * m_i_inf(vk) ** 3 * h_i * (55 - vk)
                 + g_ii * s_i * (v_rev_i - vk) + i_ext_i)
        n_inc = (n_i_inf(vk) - n_i) / tau_n_i(vk)
        h_inc = (h_i_inf(vk) - h_i) / tau_h_i(vk)
        q_inc = (1 + np.tanh(vk / 10)) / 2 * (1 - q_i) / 0.1 - q_i / tau_dq_i
        s_inc = q_i * (1 - s_i) / tau_r_i - s_i / tau_d_i

        v_tmp = vk + dt05 * v_inc
        n_tmp = n_i + dt05 * n_inc
        h_tmp = h_i + dt05 * h_inc
        q_tmp = q_i + dt05 * q_inc
        s_tmp = s_i + dt05 * s_inc

        v_inc = (0.1 * (-65 - v_tmp) + 9 * n_tmp ** 4 * (-90 - v_tmp) + 35 * m_i_inf(v_tmp) ** 3 * h_tmp * (55 - v_tmp)
                 + g_ii * s_tmp * (v_rev_i - v_tmp) + i_ext_i)
        n_inc = (n_i_inf(v_tmp) - n_tmp) / tau_n_i(v_tmp)
        h_inc = (h_i_inf(v_tmp) - h_tmp) / tau_h_i(v_tmp)
        q_inc = (1 + np.tanh(v_tmp / 10)) / 2 * (1 - q_tmp) / 0.1 - q_tmp / tau_dq_i
        s_inc = q_tmp * (1 - s_tmp) / tau_r_i - s_tmp / tau_d_i

        v_i.append(vk + dt * v_inc)
        h_i = h_i + dt * h_inc
        n_i = n_i + dt * n_inc
        q_i = q_i + dt * q_inc
        s_i = s_i + dt * s_inc

        if v_i[k + 1] < -20 and vk >= -20:
            tt = (t_old * (-20 - v_i[k + 1]) + t_new * (vk + 20)) / (vk - v_i[k + 1])
            t_i_spikes.append(tt)

    period = t_i_spikes[-1] - t_i_spikes[-2]
    v_i = np.array(v_i) if record_trace else None
    return period, v_i


def compute_condition_numbers():
    """Percentage change in the oscillation period from a 1% perturbation
    to each of i_ext, g_ii, tau_d, relative to the unperturbed base
    period. Returns (base_period, pct_i_ext, pct_g_ii, pct_tau_d, v_trace)."""
    base_period, v_trace = simulate_single_wb_ing(i_ext_i=1.5, g_ii=0.5, tau_d_i=9., record_trace=True)

    period_reduced_I, _ = simulate_single_wb_ing(i_ext_i=1.5 * 0.99, g_ii=0.5, tau_d_i=9.)
    pct_i_ext = (base_period - period_reduced_I) / base_period * 100

    period_raised_g_ii, _ = simulate_single_wb_ing(i_ext_i=1.5, g_ii=0.5 * 1.01, tau_d_i=9.)
    pct_g_ii = (base_period - period_raised_g_ii) / base_period * 100

    period_raised_tau_d, _ = simulate_single_wb_ing(i_ext_i=1.5, g_ii=0.5, tau_d_i=9. * 1.01)
    pct_tau_d = (base_period - period_raised_tau_d) / base_period * 100

    return base_period, pct_i_ext, pct_g_ii, pct_tau_d, v_trace


def plot_condition_number_trace(v_trace, dt=0.01):
    plt.figure(figsize=(8, 4))
    t = np.arange(len(v_trace)) * dt
    plt.plot(t, v_trace, '-b', linewidth=2)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
base_period, pct_i_ext, pct_g_ii, pct_tau_d, v_trace = compute_condition_numbers()
print("base_period", base_period)
print("pct_i_ext", pct_i_ext)
print("pct_g_ii", pct_g_ii)
print("pct_tau_d", pct_tau_d)
plot_condition_number_trace(v_trace)

## Abstract Pulse-Coupling Map (Inhibitory)

Reduces the population's timing to a 1-D map: $g_{\rm pulse}(\varphi)$ is
the phase shift caused by an inhibitory pulse delivered at phase
$\varphi$, $F=\varphi+g$ is the return map for one cell relative to the
pulse, and $G=F\circ F$ (applied to $1-\varphi$ twice) is the map whose
fixed points give the population's stable and unstable phase-locked
states.

In [ ]:
def H(s):
    return (1 + np.tanh(s)) / 2


def g_pulse(phi):
    return -0.5 * phi * (H((-phi + 0.8) / 0.05) - H(-4.)) * H((phi - 0.1) / 0.1)


def f_pulse(phi):
    return phi + g_pulse(phi)


def bigF_pulse(phi):
    return f_pulse(1 - phi)


def bigG_pulse(phi):
    return bigF_pulse(bigF_pulse(phi))


def simulate_abstract_pulse_coupling_inh():
    """Returns (phi, ind): a phase grid and the indices of phi where
    G(phi)-phi changes sign (candidate fixed points of G)."""
    phi = np.arange(1001) / 1000
    phi_left = phi[:1000]
    phi_right = phi[1:1001]
    G_left = bigG_pulse(phi_left)
    G_right = bigG_pulse(phi_right)
    ind = np.where((G_left - phi_left) * (G_right - phi_right) <= 0)[0]
    return phi, ind


def plot_abstract_pulse_coupling_inh(phi, ind):
    phi_left = phi[:1000]
    phi_right = phi[1:1001]
    G_left = bigG_pulse(phi_left)
    G_right = bigG_pulse(phi_right)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

    axes[0].plot(phi, g_pulse(phi), '-k', linewidth=2)
    axes[0].axis([0, 1, -1, 0])
    axes[0].set_box_aspect(1)
    axes[0].set_xlabel(r'$\varphi$')
    axes[0].set_ylabel('$g$')

    ax = axes[1]
    ax.plot(phi, bigG_pulse(phi), '-k', linewidth=2)
    ax.plot([0, 1], [0, 1], '--k', linewidth=1)
    ax.plot(phi_left[ind[0]], G_left[ind[0]], '.g', markersize=15)
    ax.plot(phi_right[ind[4]], G_right[ind[4]], '.g', markersize=15)
    ax.plot(phi_left[ind[1]], G_left[ind[1]], 'or', markersize=8, linewidth=2, fillstyle='none')
    ax.plot(phi_right[ind[3]], G_right[ind[3]], 'or', markersize=8, linewidth=2, fillstyle='none')
    ax.plot((phi_left[ind[2]] + phi_right[ind[2]]) / 2, (G_left[ind[2]] + G_right[ind[2]]) / 2,
            '.b', markersize=15)
    ax.axis([0, 1, 0, 1])
    ax.set_box_aspect(1)
    ax.set_xlabel(r'$\varphi$')
    ax.set_ylabel('$G$')

    plt.tight_layout()
    plt.show()

In [ ]:
plot_abstract_pulse_coupling_inh(*simulate_abstract_pulse_coupling_inh())

## Abstract Pulse-Coupling Map (Inhibitory), Alternative Form

A second, simpler closed-form pulse-coupling function
$g_{\rm pulse,2}(\varphi)=-\epsilon\,\varphi\,\tanh((1-\varphi)a)/\tanh(a)$,
used to explore how the shape of the PRC (via $a$) affects clustering.

In [ ]:
def g_pulse_2(phi, epsilon=0.4, a=4.0):
    return -epsilon * phi * np.tanh((1 - phi) * a) / np.tanh(a)


def simulate_abstract_pulse_coupling_inh_2():
    """Returns varphi, the phase grid on which g_pulse_2 is evaluated."""
    varphi = np.arange(1001) / 1000
    return varphi


def plot_abstract_pulse_coupling_inh_2(varphi, epsilon=0.4, a=4.0):
    plt.figure(figsize=(6, 6))
    plt.plot(varphi, g_pulse_2(varphi, epsilon=epsilon, a=a), '-k', linewidth=5)
    plt.axis([0, 1, -1, 0])
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel(r'$g(\varphi)$')
    plt.tight_layout()
    plt.show()

In [ ]:
varphi = simulate_abstract_pulse_coupling_inh_2()
interact(lambda epsilon=0.4, a=4.0: plot_abstract_pulse_coupling_inh_2(varphi, epsilon=epsilon, a=a),
         epsilon=(0.05, 0.9, 0.05), a=(0.5, 10.0, 0.5));

## ING Population Network (shared by `ING_1` through `ING_10`)

A population of `num_i` WB inhibitory cells, coupled by a random (or
fixed-indegree) inhibitory synaptic matrix and, optionally, sparse gap
junctions. `wb_init_population` splay-initializes every cell across a
random phase of its own free-running limit cycle before coupling is turned
on, exactly as in the phase-response chapters. `simulate_ing_population`
is the shared network integrator (explicit Heun stepping); `ING_1`
through `ING_10` below are all thin wrappers around it with different
heterogeneity (`sigma_i`), connection density (`p_ii`), and gap-junction
(`g_hat_gap`, `p_gap`) parameters.

In [ ]:
def wb_init_population(i_ext, phi_vec):
    """Vectorized WB single-cell integration (Heun) for each of len(i_ext)
    neurons independently, run to the 3rd spike, then (v, h, n)
    interpolated at phase phi_vec[i] between the 2nd and 3rd spikes.
    Faithfully reproduces a bug in the original source: m_tmp is computed
    from the pre-half-step v, not from v_tmp."""
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_i_inf(v)
    h = h_i_inf(v)
    n = n_i_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 9., 35., 0.1
    v_k, v_na, v_l = -90., 55., -65.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_i_inf(v) - h) / tau_h_i(v)
        n_inc = (n_i_inf(v) - n) / tau_n_i(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_i_inf(v)  # faithful port of the source's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_i_inf(v_tmp) - h_tmp) / tau_h_i(v_tmp)
        n_inc = (n_i_inf(v_tmp) - n_tmp) / tau_n_i(v_tmp)

        v = v + dt_ * v_inc
        m = m_i_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def make_g_ii_fixed_indegree(g_hat_ii, p_ii, num_i, rng):
    """Each column (post-synaptic I-cell) keeps exactly round(p_ii*num_i)
    presynaptic connections at the full weight g_hat_ii/(num_i*p_ii),
    rather than each entry independently being present with probability
    p_ii (as for the Erdos-Renyi weights built directly in
    simulate_ing_population)."""
    g_ii = g_hat_ii * np.ones((num_i, num_i)) / (num_i * p_ii)
    omit = round(num_i - p_ii * num_i)
    for j in range(num_i):
        drop = rng.choice(num_i, size=omit, replace=False)
        g_ii[drop, j] = 0.
    return g_ii


@njit
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m = 4. * math.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5.


@njit
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5.


@njit
def _ing_step_loop(m_steps, dt, dt05, num_i, v_rev_i, tau_r_i, tau_d_i, tau_dq_i,
                    i_ext_i, g_ii, G_gap, c_gap,
                    v_i, h_i, n_i, m_i, q_i, s_i):
    """Explicit-Heun per-timestep update for the I population, numba-jitted
    (population size and step count make the pure-numpy version genuinely
    slow). Matrix-vector products (synaptic and gap-junction coupling) are
    written as explicit loops since numba doesn't accelerate `@`/`.T` on
    plain ndarrays. Returns (spike times, spike indices) as numba typed
    lists."""
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)
    vi_old = np.empty(num_i)
    ii_term = np.empty(num_i)
    gap_term = np.empty(num_i)

    for step in range(m_steps):
        k = step + 1

        for i in range(num_i):
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += G_gap[i, j] * v_i[j]
            gap_term[i] = acc

        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ii_term[j] * (v_rev_i - v) + i_ext_i[j] + gap_term[j] - c_gap[j] * v)
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        for i in range(num_i):
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += G_gap[i, j] * vi_m[j]
            gap_term[i] = acc

        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ii_term[j] * (v_rev_i - v) + i_ext_i[j] + gap_term[j] - c_gap[j] * v)
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

    return i_times, i_indices


def simulate_ing_population(sigma_i, g_hat_ii, p_ii, g_hat_gap, p_gap, num_i=100, t_final=500.0,
                             dt=0.01, v_rev_i=-75.0, tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.0,
                             fixed_indegree=False, seed=63806):
    """A population of num_i synaptically and (optionally) gap-junction
    coupled WB inhibitory cells -- the shared network behind every
    ING_1..ING_10 example below. Returns (t_i_spikes, i_i_spikes).

    MATLAB's rng('default'); rng(63806) can't be bit-reproduced by NumPy,
    so results are verified structurally/statistically, not against exact
    MATLAB spike times.
    """
    rng = np.random.default_rng(seed)
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    i_ext_i = 1.5 * (1 + rng.standard_normal(num_i) * sigma_i)

    if fixed_indegree:
        g_ii = make_g_ii_fixed_indegree(g_hat_ii, p_ii, num_i, rng)
    else:
        u_ii = rng.random((num_i, num_i))
        g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)

    G_gap = np.zeros((num_i, num_i))
    for i in range(num_i - 1):
        for j in range(i + 1, num_i):
            u = rng.random()
            val = (u < p_gap) * g_hat_gap / (p_gap * (num_i - 1))
            G_gap[i, j] = val
            G_gap[j, i] = val
    c_gap = G_gap.sum(axis=1)

    iv = wb_init_population(i_ext_i, rng.random(num_i))
    v_i, h_i, n_i = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy()
    m_i = m_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    i_times, i_indices = _ing_step_loop(m_steps, dt, dt05, num_i, v_rev_i, tau_r_i, tau_d_i, tau_dq_i,
                                         i_ext_i, g_ii, G_gap, c_gap,
                                         v_i, h_i, n_i, m_i, q_i, s_i)
    t_i_spikes = np.array(i_times) if len(i_times) else np.empty(0)
    i_i_spikes = np.array(i_indices) if len(i_indices) else np.empty(0, dtype=np.int64)
    return t_i_spikes, i_i_spikes


def plot_ing_raster(t_i_spikes, i_i_spikes, num_i=100, t_final=500.0):
    plt.figure(figsize=(8, 4))
    if len(t_i_spikes) > 0:
        plt.plot(t_i_spikes, i_i_spikes, '.k', markersize=2)
    plt.yticks([1, num_i])
    plt.xlabel('$t$ [ms]')
    plt.axis([0, t_final, 0, num_i + 1])
    plt.tight_layout()
    plt.show()


def plot_ing_raster_scalebar(t_i_spikes, i_i_spikes, num_i=100, t_final=500.0):
    """Same raster, with a 100ms x 20-cell scale bracket in the
    bottom-right corner."""
    plt.figure(figsize=(8, 4))
    if len(t_i_spikes) > 0:
        plt.plot(t_i_spikes, i_i_spikes, '.k', markersize=2)
    plt.yticks([1, num_i])
    plt.xlabel('$t$ [ms]')
    plt.axis([0, t_final, 0, num_i + 1])
    plt.plot([t_final - 97, t_final], [0, 0], '-r', linewidth=8)
    plt.plot([t_final - 100, t_final], [21, 21], '-r', linewidth=4)
    plt.plot([t_final - 100, t_final - 100], [0, 21], '-r', linewidth=4)
    plt.plot([t_final, t_final], [0, 20], '-r', linewidth=8)
    plt.tight_layout()
    plt.show()


def plot_ing_raster_zoom(t_i_spikes, i_i_spikes, num_i=100, t_final=500.0):
    """Raster zoomed to the last 100ms and first 20 cells, boxed in red."""
    plt.figure(figsize=(8, 4))
    if len(t_i_spikes) > 0:
        plt.plot(t_i_spikes, i_i_spikes, '.k', markersize=2)
    plt.yticks([1, 20])
    plt.xlabel('$t$ [ms]')
    plt.axis([t_final - 100, t_final, 0, 21])
    plt.plot([t_final - 100, t_final, t_final, t_final - 100, t_final - 100],
             [0, 0, 21, 21, 0], '-r', linewidth=4)
    plt.tight_layout()
    plt.show()

## ING_1

The population reference raster: homogeneous drive (`sigma_i=0`), an
all-to-all inhibitory network (`p_ii=1`), no gap junctions -- the same
network configuration revisited with a scale bar in `ING_7` below. (Ports
the same Heun-integrated network used throughout this notebook rather than
the legacy `odeint`+`networkx` script, whose splay-state initializer had a
bug and never actually ran.) Drag `g_hat_ii` to see how synaptic strength
sets how tightly the population clusters.

In [ ]:
def simulate_ing_1(g_hat_ii=0.5):
    return simulate_ing_population(sigma_i=0.0, g_hat_ii=g_hat_ii, p_ii=1., g_hat_gap=0.0, p_gap=1.)

In [ ]:
interact(lambda g_hat_ii=0.5: plot_ing_raster(*simulate_ing_1(g_hat_ii=g_hat_ii)),
         g_hat_ii=(0.0, 1.0, 0.05));

## ING_2

Mild drive heterogeneity across the population (`sigma_i=0.03`), otherwise
the same all-to-all network as `ING_1`. Drag `sigma_i` to see how much
heterogeneity the network can tolerate before synchrony breaks down.

In [ ]:
def simulate_ing_2(sigma_i=0.03):
    return simulate_ing_population(sigma_i=sigma_i, g_hat_ii=0.5, p_ii=1., g_hat_gap=0.0, p_gap=1.)

In [ ]:
interact(lambda sigma_i=0.03: plot_ing_raster(*simulate_ing_2(sigma_i=sigma_i)),
         sigma_i=(0.0, 0.2, 0.01));

## ING_3

Homogeneous drive but sparser inhibitory connectivity (`p_ii=0.85`,
Erdos-Renyi). Drag `p_ii` to see how sparse the synaptic wiring can get
before the population desynchronizes.

In [ ]:
def simulate_ing_3(p_ii=0.85):
    return simulate_ing_population(sigma_i=0.0, g_hat_ii=0.5, p_ii=p_ii, g_hat_gap=0.0, p_gap=1.)

In [ ]:
interact(lambda p_ii=0.85: plot_ing_raster(*simulate_ing_3(p_ii=p_ii)),
         p_ii=(0.1, 1.0, 0.05));

## ING_4

Same connection density as `ING_3` (`p_ii=0.85`), but with exactly
`round(p_ii*num_i)` presynaptic partners per cell (`fixed_indegree=True`)
instead of independent Erdos-Renyi wiring. Drag `p_ii` to compare against
`ING_3`'s Erdos-Renyi wiring at the same density.

In [ ]:
def simulate_ing_4(p_ii=0.85):
    return simulate_ing_population(sigma_i=0.0, g_hat_ii=0.5, p_ii=p_ii, g_hat_gap=0.0, p_gap=1.,
                                    fixed_indegree=True)

In [ ]:
interact(lambda p_ii=0.85: plot_ing_raster(*simulate_ing_4(p_ii=p_ii)),
         p_ii=(0.1, 1.0, 0.05));

## ING_5

Stronger drive heterogeneity (`sigma_i=0.05`) and sparser connectivity
(`p_ii=0.5`). Drag `sigma_i` to see how much drive heterogeneity a sparser
network can tolerate compared to `ING_2`'s all-to-all network.

In [ ]:
def simulate_ing_5(sigma_i=0.05):
    return simulate_ing_population(sigma_i=sigma_i, g_hat_ii=0.5, p_ii=0.5, g_hat_gap=0.0, p_gap=1.)

In [ ]:
interact(lambda sigma_i=0.05: plot_ing_raster(*simulate_ing_5(sigma_i=sigma_i)),
         sigma_i=(0.0, 0.2, 0.01));

## ING_6

Same heterogeneity and sparsity as `ING_5`, plus sparse gap junctions
(`g_hat_gap=0.1`, `p_gap=0.05`) to see whether electrical coupling
resynchronizes what the sparse/heterogeneous synaptic network alone could
not. Drag `g_hat_gap` to see how much electrical coupling is needed to
recover synchrony.

In [ ]:
def simulate_ing_6(g_hat_gap=0.1):
    return simulate_ing_population(sigma_i=0.05, g_hat_ii=0.5, p_ii=0.5, g_hat_gap=g_hat_gap, p_gap=0.05)

In [ ]:
interact(lambda g_hat_gap=0.1: plot_ing_raster(*simulate_ing_6(g_hat_gap=g_hat_gap)),
         g_hat_gap=(0.0, 0.3, 0.01));

## ING_7

The `ING_1` configuration again (homogeneous, all-to-all, no gap
junctions), now with a 100ms x 20-cell scale bar drawn on the raster.
Drag `tau_d_i` (the inhibitory decay time constant) to see how it sets
the population's oscillation period.

In [ ]:
def simulate_ing_7(tau_d_i=9.0):
    return simulate_ing_population(sigma_i=0.0, g_hat_ii=0.5, p_ii=1., g_hat_gap=0.0, p_gap=1.,
                                    tau_d_i=tau_d_i)

In [ ]:
interact(lambda tau_d_i=9.0: plot_ing_raster_scalebar(*simulate_ing_7(tau_d_i=tau_d_i)),
         tau_d_i=(2.0, 15.0, 0.5));

## ING_8

Homogeneous, all-to-all synaptic network, plus very sparse gap junctions
(`p_gap=0.05`) with electrical coupling strength left at 0 -- zoomed to
the last 100ms and first 20 cells. Drag `g_hat_gap` up from 0 to turn on
electrical coupling in this sparse-gap topology.

In [ ]:
def simulate_ing_8(g_hat_gap=0.0):
    return simulate_ing_population(sigma_i=0.0, g_hat_ii=0.5, p_ii=1., g_hat_gap=g_hat_gap, p_gap=5 / 100)

In [ ]:
interact(lambda g_hat_gap=0.0: plot_ing_raster_zoom(*simulate_ing_8(g_hat_gap=g_hat_gap)),
         g_hat_gap=(0.0, 0.3, 0.01));

## ING_9

Like `ING_8`, with drive heterogeneity (`sigma_i=0.05`) added back in.
Drag `sigma_i` to see how heterogeneity interacts with sparse gap-junction
coupling.

In [ ]:
def simulate_ing_9(sigma_i=0.05):
    return simulate_ing_population(sigma_i=sigma_i, g_hat_ii=0.5, p_ii=1., g_hat_gap=0.0, p_gap=5 / 100)

In [ ]:
interact(lambda sigma_i=0.05: plot_ing_raster_zoom(*simulate_ing_9(sigma_i=sigma_i)),
         sigma_i=(0.0, 0.2, 0.01));

## ING_10

Like `ING_9`, now with gap junctions actually turned on
(`g_hat_gap=0.04`, `p_gap=0.05`) -- the full heterogeneous,
sparsely-synaptic-plus-electrically-coupled network. Drag `g_hat_gap` to
see how much electrical coupling this heterogeneous network needs to
resynchronize.

In [ ]:
def simulate_ing_10(g_hat_gap=0.04):
    return simulate_ing_population(sigma_i=0.05, g_hat_ii=0.5, p_ii=1., g_hat_gap=g_hat_gap, p_gap=0.05)

In [ ]:
interact(lambda g_hat_gap=0.04: plot_ing_raster_zoom(*simulate_ing_10(g_hat_gap=g_hat_gap)),
         g_hat_gap=(0.0, 0.3, 0.01));

## ING Entraining E Cells

A population of RTM excitatory cells receives inhibition from the WB
inhibitory population above (I -> E), while I cells are also
gap-junction coupled to each other; there is no E -> I or E -> E coupling
here (`g_hat_ee = g_hat_ei = 0`), so the E cells only ever see the
inhibitory rhythm, not each other. `rtm_init_population` splay-initializes
the E cells the same way `wb_init_population` does the I cells.
`build_entrainment_network` builds the (fixed) synaptic and gap-junction
wiring; `simulate_entrainment` runs the coupled network for a given E-cell
drive. Drag `g_hat_ie` (I->E coupling strength) to see how strongly the
inhibitory rhythm needs to drive E cells before it entrains them.

In [ ]:
def m_e_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - np.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (np.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


def h_e_inf(v):
    alpha_h = 0.128 * np.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + np.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


def tau_h_e(v):
    alpha_h = 0.128 * np.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + np.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


def n_e_inf(v):
    alpha_n = 0.032 * (v + 52) / (1 - np.exp(-(v + 52) / 5))
    beta_n = 0.5 * np.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


def tau_n_e(v):
    alpha_n = 0.032 * (v + 52) / (1 - np.exp(-(v + 52) / 5))
    beta_n = 0.5 * np.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


def rtm_init_population(i_ext, phi_vec):
    """Vectorized RTM single-cell integration (Heun), analogous to
    wb_init_population, for E cells."""
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of the source's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def build_entrainment_network(num_e=400, num_i=100, sigma_e=0.10, sigma_i=0.05, drive_e=1.5,
                               drive_i=1.5, g_hat_ee=0.0, g_hat_ei=0.0, g_hat_ie=0.5, g_hat_ii=0.5,
                               g_hat_gap=0.1, p_gap=0.05, p_ee=1.0, p_ei=0.5, p_ie=0.5, p_ii=0.5,
                               tau_d_e=3.0, tau_r_e=0.5, tau_peak_e=0.5,
                               tau_d_i=9.0, tau_r_i=0.5, tau_peak_i=0.5, seed=63806):
    """Builds a fixed E-I network (random synaptic + sparse gap-junction
    wiring): E cells are RTM, I cells are WB, and only the I population is
    gap-junction coupled. Returns a dict with everything simulate_entrainment
    needs, including the still-live rng (further calls continue drawing
    from the same stream, as in the original scripts)."""
    rng = np.random.default_rng(seed)

    i_ext_e = drive_e * (1 + sigma_e * rng.standard_normal(num_e))
    i_ext_i = drive_i * (1 + sigma_i * rng.standard_normal(num_i))

    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    u_ee = rng.random((num_e, num_e))
    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ee = g_hat_ee * (u_ee < p_ee) / (num_e * p_ee)
    g_ei = g_hat_ei * (u_ei < p_ei) / (num_e * p_ei)
    g_ie = g_hat_ie * (u_ie < p_ie) / (num_i * p_ie)
    g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)

    G_gap = np.zeros((num_i, num_i))
    for i in range(num_i - 1):
        for j in range(i + 1, num_i):
            u = rng.random()
            val = (u < p_gap) * g_hat_gap / (p_gap * (num_i - 1))
            G_gap[i, j] = val
            G_gap[j, i] = val
    c_gap = G_gap.sum(axis=1)

    return dict(rng=rng, num_e=num_e, num_i=num_i, i_ext_e=i_ext_e, i_ext_i=i_ext_i,
                g_ee=g_ee, g_ei=g_ei, g_ie=g_ie, g_ii=g_ii, G_gap=G_gap, c_gap=c_gap,
                tau_dq_e=tau_dq_e, tau_dq_i=tau_dq_i, tau_r_e=tau_r_e, tau_d_e=tau_d_e,
                tau_r_i=tau_r_i, tau_d_i=tau_d_i, v_rev_e=0.0, v_rev_i=-75.0)


@njit
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


@njit
def _entrainment_step_loop(m_steps, dt, dt05, num_e, num_i,
                            v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                            tau_r_i, tau_d_i, tau_dq_i,
                            i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii, G_gap, c_gap,
                            v_e, h_e, n_e, m_e, q_e, s_e,
                            v_i, h_i, n_i, m_i, q_i, s_i):
    """Explicit-Heun per-timestep update for the coupled E-I network,
    numba-jitted for the same reason as _ing_step_loop above."""
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)
    gap_term = np.empty(num_i)

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_i[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += G_gap[i, j] * v_i[j]
            gap_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v)
                      + i_ext_i[j] + gap_term[j] - c_gap[j] * v)
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * si_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += G_gap[i, j] * vi_m[j]
            gap_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v)
                      + i_ext_i[j] + gap_term[j] - c_gap[j] * v)
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

    return e_times, e_indices, i_times, i_indices


def simulate_entrainment(net, i_ext_e_local, t_final=500.0, dt=0.01):
    """Runs the shared E-I network in `net` (from build_entrainment_network)
    with a given E-cell drive vector. Returns (t_e_spikes, i_e_spikes,
    t_i_spikes, i_i_spikes)."""
    rng = net["rng"]
    num_e, num_i = net["num_e"], net["num_i"]
    g_ee, g_ei, g_ie, g_ii = net["g_ee"], net["g_ei"], net["g_ie"], net["g_ii"]
    G_gap, c_gap = net["G_gap"], net["c_gap"]
    tau_dq_e, tau_dq_i = net["tau_dq_e"], net["tau_dq_i"]
    tau_r_e, tau_d_e = net["tau_r_e"], net["tau_d_e"]
    tau_r_i, tau_d_i = net["tau_r_i"], net["tau_d_i"]
    v_rev_e, v_rev_i = net["v_rev_e"], net["v_rev_i"]
    i_ext_i = net["i_ext_i"]

    dt05 = dt / 2
    m_steps = round(t_final / dt)

    iv = rtm_init_population(i_ext_e_local, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy()
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    iv = wb_init_population(i_ext_i, rng.random(num_i))
    v_i, h_i, n_i = iv[:, 0].copy(), iv[:, 1].copy(), iv[:, 2].copy()
    m_i = m_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    e_times, e_indices, i_times, i_indices = _entrainment_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i,
        i_ext_e_local, i_ext_i, g_ee, g_ei, g_ie, g_ii, G_gap, c_gap,
        v_e, h_e, n_e, m_e, q_e, s_e,
        v_i, h_i, n_i, m_i, q_i, s_i,
    )
    t_e_spikes = np.array(e_times) if len(e_times) else np.empty(0)
    i_e_spikes = np.array(e_indices) if len(e_indices) else np.empty(0, dtype=np.int64)
    t_i_spikes = np.array(i_times) if len(i_times) else np.empty(0)
    i_i_spikes = np.array(i_indices) if len(i_indices) else np.empty(0, dtype=np.int64)
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes


def simulate_ing_entraining_e_cells(g_hat_ie=0.50):
    """The ING-entrains-E-cells reference configuration. Returns
    (t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, num_e, num_i)."""
    net = build_entrainment_network(sigma_e=0.10, sigma_i=0.05, drive_e=1.5, drive_i=1.5,
                                     g_hat_ie=g_hat_ie, g_hat_ii=0.50, g_hat_gap=0.1, p_gap=0.05,
                                     p_ee=1.0, p_ei=0.5, p_ie=0.5, p_ii=0.5)
    t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes = simulate_entrainment(net, net["i_ext_e"])
    return t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, net["num_e"], net["num_i"]


def plot_entrainment_raster(t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes,
                             num_e=400, num_i=100, t_final=500.0):
    plt.figure(figsize=(8, 4))
    if len(t_i_spikes) > 0:
        plt.plot(t_i_spikes, i_i_spikes, '.b', markersize=2)
    if len(t_e_spikes) > 0:
        plt.plot(t_e_spikes, i_e_spikes + num_i, '.r', markersize=2)
    plt.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
    plt.yticks([num_i, num_e + num_i])
    plt.xlabel('$t$ [ms]')
    plt.axis([0, t_final, 0, num_e + num_i + 1])
    plt.tight_layout()
    plt.show()

In [ ]:
def _plot_entrain_e_cells(g_hat_ie=0.50):
    t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, num_e, num_i = simulate_ing_entraining_e_cells(
        g_hat_ie=g_hat_ie)
    plot_entrainment_raster(t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, num_e=num_e, num_i=num_i)

In [ ]:
interact(_plot_entrain_e_cells, g_hat_ie=(0.0, 1.0, 0.05));

## ING Entraining E Cells 2

Same network topology as above but all-to-all E-I/I-E/I-I connectivity
(`p_ee=p_ei=p_ie=p_ii=1`) and homogeneous drive, swept over three E-cell
drive levels (`i_ext_e_vec`) to see how the fraction of E cells recruited
into each inhibitory window grows with drive. `run_drive_panels` builds
the network once and simulates each drive level in turn (continuing the
same rng stream for phase-splay draws, as the original script did);
`main` renders and saves the three-panel figure.

In [ ]:
def run_drive_panels(t_final_run=500.0, i_ext_e_vec=(1.9, 2.0, 2.1), seed=63806):
    """Builds one E-I network and simulates it once per drive level in
    i_ext_e_vec. Returns a list of (t_e_spikes, i_e_spikes, t_i_spikes,
    i_i_spikes) tuples, one per drive."""
    net = build_entrainment_network(sigma_e=0.0, sigma_i=0.0, drive_e=1.9, drive_i=1.5,
                                     g_hat_ie=0.5, g_hat_ii=0.5, g_hat_gap=0.1, p_gap=0.05,
                                     p_ee=1.0, p_ei=1.0, p_ie=1.0, p_ii=1.0, seed=seed)
    rng = net["rng"]
    num_e = net["num_e"]
    return [
        simulate_entrainment(net, drive * np.ones(num_e) * (1 + 0.0 * rng.standard_normal(num_e)),
                              t_final=t_final_run)
        for drive in i_ext_e_vec
    ]


def main():
    i_ext_e_vec = (1.9, 2.0, 2.1)
    num_e, num_i, t_final = 400, 100, 500.0
    results = run_drive_panels()

    fig, axes = plt.subplots(3, 1, figsize=(8, 9))
    for ax, i_ext_e_val, (t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes) in zip(axes, i_ext_e_vec, results):
        if len(t_i_spikes) > 0:
            ax.plot(t_i_spikes, i_i_spikes, '.b', markersize=2)
        if len(t_e_spikes) > 0:
            ax.plot(t_e_spikes, i_e_spikes + num_i, '.r', markersize=2)
        ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
        ax.set_yticks([num_i, num_e + num_i])
        ax.axis([0, t_final, 0, num_e + num_i + 1])
        ax.set_title(rf'$\overline{{I}}_E={i_ext_e_val:g}$')
    axes[-1].set_xlabel('$t$ [ms]')
    plt.tight_layout()
    plt.savefig("fig.png")
    plt.show()

In [ ]:
main()